# COMPACT-MI-RFE — CICFLOW Colab Orchestrator

This notebook is a new CICFLOW orchestrator modeled on the purpose of `COMPACT_MI_RFE_ColabPro.ipynb`: Runtime → Drive → Dataset source → Source checkout → Dependencies/tests → Dataset audit → Experiment selection → Preflight → Isolated run → Verify → Summary/backup. It does not modify the reference notebook, the CTU-13 scripts, the Windows dataset, or completed Stage 4 runs.

Run the authoritative CICFLOW cells headed `CELL 1` through `CELL 12` in order. The old scaffold cells retained below the new sequence are legacy reference material and must not be run. Default mode is `dry_run`. MI/RFE fitting, resume, and Test evaluation are never triggered by the default configuration. Stage 4/5 remain `PROVISIONAL_RESEARCH` until the reference contract is reconciled.

## Scientific and data rules

- One dataset root only: `CICFLOWMETER_FEATURE_CSV`.
- Four configurations are supported.
- MI and RFE read Train only. Validation/Test are not opened in these stages.
- Stage 4 MI is bound to its manifest and identity digest. Stage 5A binds to the exact MI run.
- XGBoost-RFE is blocked until an estimator/ranking contract is verified.
- The notebook does not silently change StandardScaler, MI seeds, estimator parameters, or tie semantics.

In [ ]:
# CELL 1 — Runtime information and safe configuration
import importlib.metadata as im
import json, os, platform, shutil, sys
try:
    import psutil
    vm = psutil.virtual_memory()
    RAM_INFO = {'total': vm.total, 'available': vm.available}
    PHYSICAL_CPUS = psutil.cpu_count(logical=False)
except ImportError:
    RAM_INFO, PHYSICAL_CPUS = {'total': None, 'available': None}, None
DISK_INFO = shutil.disk_usage('/content')
GPU_AVAILABLE = shutil.which('nvidia-smi') is not None
VERSIONS = {}
for package in ('numpy', 'pandas', 'scipy', 'scikit-learn', 'pyarrow', 'psutil'):
    try: VERSIONS[package] = im.version(package)
    except im.PackageNotFoundError: VERSIONS[package] = None
print({'os': platform.platform(), 'python': sys.version, 'logical_cpus': os.cpu_count(),
       'physical_cpus': PHYSICAL_CPUS, 'ram_bytes': RAM_INFO,
       'disk_free_bytes': DISK_INFO.free, 'gpu_probe_available': GPU_AVAILABLE,
       'versions': VERSIONS})
print('MI/RFE uses CPU sklearn paths; GPU acceleration is not assumed.')

# Safe defaults: change explicitly before running a non-dry mode.
MODE = 'dry_run'  # dry_run | full | resume | verify
REFERENCE_RECONCILIATION_CONFIRMED = False
CONFIRM_PROVISIONAL_RESEARCH = False
INSTALL_DEPENDENCIES = False
MOUNT_DRIVE = False
BACKUP_SMALL_ARTIFACTS = False
GIT_COMMIT = 'NOT_CAPTURED'
EXISTING_RESULT_ROOT = ''  # required only for resume/verify

DRIVE_ROOT = '/content/drive/MyDrive'
PROJECT_ROOT = '/content/drive/MyDrive'
SOURCE_ROOT = '/content/drive/MyDrive/code_CICFLOW'
DATASET_ROOT = '/content/drive/MyDrive/CICFLOWMETER_FEATURE_CSV'

CONFIGS = ['FW256_FS8', 'FW256_FS16', 'FW512_FS8', 'FW512_FS16']
BRANCHES = ['rf', 'svm']
RUN_IDS = {}  # populated once the fresh Colab result root is created

if MODE not in {'dry_run', 'full', 'resume', 'verify'}:
    raise ValueError(f'Unsupported MODE: {MODE}')
if MODE in {'full', 'resume'} and not REFERENCE_RECONCILIATION_CONFIRMED:
    raise RuntimeError('Set REFERENCE_RECONCILIATION_CONFIRMED=True only after source reconciliation')
if MODE in {'full', 'resume'} and not CONFIRM_PROVISIONAL_RESEARCH:
    raise RuntimeError('Set CONFIRM_PROVISIONAL_RESEARCH=True for provisional full/resume runs')

## CELL 2 — Mount Google Drive and setup source paths

Run this cell only when intentionally mounting the Drive. It does not clone a repository or install packages by default. The source checkout and dataset package must be prepared by the user without copying unnecessary Test files.

In [ ]:
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

if INSTALL_DEPENDENCIES:
    # Pin versions only after reconciling them with the reference environment.
    get_ipython().run_line_magic('pip', 'install numpy==1.26.4 pandas==2.2.3 pyarrow==18.1.0 scikit-learn==1.5.2 psutil==6.1.0')

from pathlib import Path
import sys
DRIVE_ROOT = Path(DRIVE_ROOT)
PROJECT_ROOT = Path(PROJECT_ROOT)
SOURCE_ROOT = Path(SOURCE_ROOT)
DATASET_ROOT = Path(DATASET_ROOT)
sys.path.insert(0, str(SOURCE_ROOT))

from colab.colab_pipeline import (
    BRANCHES, CONFIGS, build_paths, check_source_layout,
    stage4_metadata_preflight, summarize_configs,
    verify_stage4_run, run_stage4_mi, run_stage5_rfe,
    backup_small_artifacts, source_identity_sha256,
)
if MODE in {'resume', 'verify'}:
    if not EXISTING_RESULT_ROOT:
        raise RuntimeError('Set EXISTING_RESULT_ROOT for resume/verify; no fresh directory is created for those modes')
    RESULT_ROOT = Path(EXISTING_RESULT_ROOT)
    if not RESULT_ROOT.is_dir():
        raise FileNotFoundError(RESULT_ROOT)
    RUN_ID = RESULT_ROOT.name
else:
    RUN_STAMP = __import__('datetime').datetime.now(__import__('datetime').timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
    RUN_ID = f'RUN_MI_{RUN_STAMP}'
    RESULT_ROOT = DRIVE_ROOT / 'COMPACT_MI_RFE_FRESH_RUNS' / RUN_ID
    if RESULT_ROOT.exists():
        raise FileExistsError(f'Fresh result root already exists: {RESULT_ROOT}')
    (RESULT_ROOT / 'stage4_mi' / 'runs').mkdir(parents=True, exist_ok=False)
    (RESULT_ROOT / 'stage5_rfe' / 'runs').mkdir(parents=True, exist_ok=False)
RUN_IDS = {config: RUN_ID for config in CONFIGS}
paths = build_paths(project_root=PROJECT_ROOT, source_root=SOURCE_ROOT, dataset_root=DATASET_ROOT, drive_root=DRIVE_ROOT, result_root=RESULT_ROOT)
SOURCE_SHA256 = source_identity_sha256(SOURCE_ROOT)
layout_status = check_source_layout(paths)
print('=== GOOGLE DRIVE PATH CHECK ===')
for label, path in [('DRIVE_ROOT', DRIVE_ROOT), ('PROJECT_ROOT', PROJECT_ROOT), ('SOURCE_ROOT', SOURCE_ROOT), ('DATASET_ROOT', DATASET_ROOT)]:
    print(f'{label} {"FOUND" if path.exists() else "MISSING"}')
    print(f'  {path}')
print('\n=== SOURCE CODE CHECK ===')
for relative in ('colab/colab_pipeline.py', 'stage4_mi/mi_selection.py', 'stage5_rfe/rfe_selection.py'):
    path = SOURCE_ROOT / relative
    print(('PASS' if path.is_file() else 'FAIL'), path)
print('\n=== DATASET CONFIGURATION CHECK ===')
for config in CONFIGS:
    print(config, 'FOUND' if (DATASET_ROOT / config).is_dir() else 'MISSING')
print('\n=== FIX PYTHON IMPORT ===')
print('Expected:', SOURCE_ROOT / 'colab/colab_pipeline.py')
print('Loaded:  ', __import__('colab.colab_pipeline', fromlist=['__file__']).__file__)
print('\n=== PIPELINE IMPORT SUCCESS ===')
print('Loaded from:', __import__('colab.colab_pipeline', fromlist=['__file__']).__file__)
print('CONFIGS:', CONFIGS)
print('BRANCHES:', BRANCHES)
print('\n=== SOURCE IDENTITY ===')
print('SHA256:', SOURCE_SHA256)
print('\n=== FRESH OUTPUT INITIALIZATION ===')
print('CREATED:', RESULT_ROOT / 'stage4_mi' / 'runs')
print('CREATED:', RESULT_ROOT / 'stage5_rfe' / 'runs')
print('\n=== SOURCE LAYOUT CHECK ===')
print(json.dumps(layout_status, indent=2, default=str))
print('\n=== SETUP COMPLETED ===')
print('SOURCE_ROOT:', SOURCE_ROOT)
print('DATASET_ROOT:', DATASET_ROOT)
print('RESULT_ROOT:', RESULT_ROOT)
print('Stage 4:', paths['stage4_runs_root'])
print('Stage 5:', paths['stage5_runs_root'])
print('All four dataset configurations found.')
print('Fresh output directories created.')
print('No MI/RFE training executed.')

## CELL 3 — Dataset source

The CICFLOW notebook uses one mounted `CICFLOWMETER_FEATURE_CSV` root. It does not download, extract, merge, or copy another dataset root.

In [ ]:
print('=== CELL 3 — CICFLOW DATASET SOURCE ===')
print('DATASET_ROOT:', paths['dataset_root'])
print('Exists:', paths['dataset_root'].is_dir())
if not paths['dataset_root'].is_dir():
    raise FileNotFoundError(paths['dataset_root'])
print('Configurations:', [item.name for item in paths['dataset_root'].iterdir() if item.is_dir()])
print('No dataset copied or downloaded by this cell.')

## CELL 4 — Safe source checkout

The GitHub checkout is inspected only. Existing checkout changes are not reset, pulled, or overwritten. The notebook source and Stage 4/5 modules must be present before a non-dry mode.

In [ ]:
CLONE_SOURCE = False
SOURCE_URL = ''  # Set explicitly before enabling CLONE_SOURCE
if CLONE_SOURCE:
    import subprocess
    if (PROJECT_ROOT / '.git').is_dir():
        status = subprocess.run(['git', '-C', str(PROJECT_ROOT), 'status', '--porcelain'], text=True, capture_output=True, check=True)
        print('Existing checkout; no pull/reset. Local changes:', status.stdout.splitlines())
    elif SOURCE_URL:
        subprocess.run(['git', 'clone', '--', SOURCE_URL, str(PROJECT_ROOT)], check=True)
    else:
        raise ValueError('Set SOURCE_URL before enabling CLONE_SOURCE')

SOURCE_READY = all((SOURCE_ROOT / relative).exists() for relative in (
    'stage4_mi/mi_selection.py',
    'stage5_rfe/rfe_selection.py',
    'colab/colab_pipeline.py',
))
print('Source checkout:', PROJECT_ROOT)
print('Required CICFLOW modules present:', SOURCE_READY)
if not SOURCE_READY:
    raise FileNotFoundError('Upload/clone the selected CICFLOW source before continuing')

## CELL 5 — Dependencies and tests

This cell is opt-in. It verifies the source and can run synthetic unit tests; it never starts MI/RFE training.

In [ ]:
RUN_TESTS = False
TEST_RESULTS = {'status': 'SKIPPED'}
if RUN_TESTS:
    import subprocess
    commands = [
        [sys.executable, '-m', 'unittest', 'discover', '-s', str(PROJECT_ROOT / 'code_CICFLOW/colab/tests'), '-p', 'test_*.py', '-v'],
        [sys.executable, '-m', 'unittest', 'discover', '-s', str(PROJECT_ROOT / 'code_CICFLOW/stage4_mi/tests'), '-p', 'test_*.py', '-v'],
        [sys.executable, '-m', 'unittest', 'discover', '-s', str(PROJECT_ROOT / 'code_CICFLOW/stage5_rfe/tests'), '-p', 'test_*.py', '-v'],
    ]
    for command in commands:
        completed = subprocess.run(command, cwd=str(PROJECT_ROOT / 'code_CICFLOW'), text=True, capture_output=True)
        print(completed.stdout[-4000:]); print(completed.stderr[-8000:])
        if completed.returncode != 0:
            raise RuntimeError(f'Unit test command failed: {command}')
    TEST_RESULTS = {'status': 'PASS'}
print('CELL 5:', TEST_RESULTS)

## CELL 6 — Dataset/schema audit

Metadata-only audit for all four configurations. It checks Train footer, schema, row counts, and resources without loading feature values.

In [ ]:
DATA_AUDIT = {}
print('layout_status:', layout_status)
print('SOURCE_READY:', SOURCE_READY)
if layout_status['status'] == 'PASS' and SOURCE_READY:
    import os, subprocess
    audit_env = os.environ.copy()
    audit_env['PYTHONPATH'] = str(SOURCE_ROOT)
    for config in CONFIGS:
        command = [sys.executable, '-m', 'stage4_mi.mi_preflight', '--mode', 'metadata', '--config', config, '--dataset-root', str(DATASET_ROOT), '--runs-root', str(paths['stage4_runs_root']), '--run-id', RUN_IDS[config]]
        completed = subprocess.run(command, cwd=str(SOURCE_ROOT), env=audit_env, capture_output=True, text=True)
        print(completed.stdout[-4000:])
        print(completed.stderr[-8000:])
        if not completed.stdout.strip():
            raise RuntimeError(f'Metadata worker returned no JSON for {config}')
        DATA_AUDIT[config] = json.loads(completed.stdout)
else:
    print('DATA_AUDIT skipped because layout/source gate is not PASS.')
print(json.dumps(DATA_AUDIT, indent=2, default=str))

## CELL 7 — Validate CICFLOW dataset readiness

This is the explicit readiness gate corresponding to the reference notebook's dataset validation cell. It uses the metadata results from Cell 6 and never opens Validation/Test values.

In [ ]:
CICFLOW_DATASET_READY = bool(DATA_AUDIT) and all(result.get('status') == 'PASS' for result in DATA_AUDIT.values())
print('CICFLOW_DATASET_READY:', CICFLOW_DATASET_READY)
print('Validation/Test feature rows read: 0')
if MODE in {'full', 'resume'} and not CICFLOW_DATASET_READY:
    raise RuntimeError('CICFLOW dataset readiness gate failed; MI/RFE blocked')

## CELL 8 — Select experiment/configuration

The reference notebook selects one experiment/configuration before preflight. This CICFLOW version keeps the four configurations explicit and uses the supplied Stage 4 Run IDs.

In [ ]:
SELECTED_CONFIG = None  # None means the four-configuration summary
SELECTED_BRANCHES = ['rf', 'svm']
if SELECTED_CONFIG is not None and SELECTED_CONFIG not in CONFIGS:
    raise ValueError(f'Unknown configuration: {SELECTED_CONFIG}')
print({'mode': MODE, 'selected_config': SELECTED_CONFIG, 'branches': SELECTED_BRANCHES, 'run_ids': RUN_IDS, 'scientific_status': 'PROVISIONAL_RESEARCH'})

## CELL 9 — Safe preflight

Preflight must pass before full MI/RFE. It does not call `model.fit`, does not read Validation/Test, and does not change parameters automatically.

In [ ]:
PREFLIGHT = DATA_AUDIT
print('=== CELL 9 PREFLIGHT ===')
for config, result in PREFLIGHT.items():
    print(config, result.get('status'), result.get('errors', []))
if MODE in {'full', 'resume'}:
    blocked = [config for config, result in PREFLIGHT.items() if result.get('status') != 'PASS']
    if blocked:
        raise RuntimeError(f'Preflight BLOCKED for: {blocked}')
print('No MI/RFE fit started by preflight.')

## CELL 10 — Isolated MI/RFE run

This is the only cell that can start MI/RFE, and only after explicit mode/confirmation flags. `dry_run` prints the guard and starts no fit.

In [ ]:
import subprocess
RUN_RESULTS = {}
configs_to_run = [SELECTED_CONFIG] if SELECTED_CONFIG else CONFIGS
run_env = os.environ.copy()
run_env['PYTHONPATH'] = str(SOURCE_ROOT)
def run_isolated(label, command):
    print(f'\n=== {label} ===')
    print('COMMAND:', ' '.join(command))
    process = subprocess.Popen(command, cwd=str(SOURCE_ROOT), env=run_env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    lines = []
    for line in process.stdout:
        print(line, end='', flush=True)
        lines.append(line)
    exit_code = process.wait()
    if exit_code != 0:
        raise RuntimeError(f'{label} failed with exit code {exit_code}; preserve incomplete artifacts')
    return {'status': 'PROGRAM_SUCCEEDED', 'exit_code': exit_code, 'output_tail': ''.join(lines[-20:])}
if MODE in {'full', 'resume'}:
    if not CONFIRM_PROVISIONAL_RESEARCH or not REFERENCE_RECONCILIATION_CONFIRMED:
        raise RuntimeError('Full/resume requires both explicit confirmation flags')
    for config in configs_to_run:
        run_id = RUN_IDS[config]
        mi_command = [sys.executable, '-m', 'stage4_mi.mi_selection', '--mode', MODE, '--config', config, '--dataset-root', str(DATASET_ROOT), '--runs-root', str(paths['stage4_runs_root']), '--run-id', run_id, '--confirm-provisional-research']
        RUN_RESULTS[f'{config}/mi'] = run_isolated(f'STAGE 4 MI {config}', mi_command)
        for branch in SELECTED_BRANCHES:
            rfe_command = [sys.executable, '-m', 'stage5_rfe.rfe_selection', '--mode', MODE, '--branch', branch, '--config', config, '--dataset-root', str(DATASET_ROOT), '--mi-runs-root', str(paths['stage4_runs_root']), '--runs-root', str(paths['stage5_runs_root']), '--run-id', run_id, '--git-commit', GIT_COMMIT, '--confirm-provisional-research']
            RUN_RESULTS[f'{config}/{branch}'] = run_isolated(f'STAGE 5A {config}/{branch}', rfe_command)
elif MODE == 'verify':
    for config in configs_to_run:
        run_id = RUN_IDS[config]
        RUN_RESULTS[f'{config}/mi'] = run_isolated(f'VERIFY STAGE 4 {config}', [sys.executable, '-m', 'stage4_mi.mi_selection', '--mode', 'verify', '--run-dir', str(paths['stage4_runs_root'] / config / run_id)])
        for branch in SELECTED_BRANCHES:
            RUN_RESULTS[f'{config}/{branch}'] = run_isolated(f'VERIFY STAGE 5A {config}/{branch}', [sys.executable, '-m', 'stage5_rfe.rfe_selection', '--mode', 'verify', '--run-dir', str(paths['stage5_runs_root'] / config / run_id / branch)])
else:
    RUN_RESULTS = {'status': 'DRY_RUN_ONLY', 'model_fit_calls': 0, 'test_feature_rows_read': 0}
print(json.dumps(RUN_RESULTS, indent=2, default=str))

## CELL 11 — Verify completed artifacts

Verification reads result JSON/markers only. It does not rerun MI/RFE and does not open Test data.

In [ ]:
VERIFY_RESULTS = {}
if MODE == 'verify':
    VERIFY_RESULTS = RUN_RESULTS
else:
    VERIFY_RESULTS = {'status': 'NOT_RUN', 'reason': 'Set MODE=verify explicitly'}
print(json.dumps(VERIFY_RESULTS, indent=2, default=str))

## CELL 12 — Summary and small-artifact backup

The summary follows the reference notebook's result-only publication step. Backup is opt-in and copies no Parquet.

In [ ]:
SUMMARY = summarize_configs(paths, RUN_IDS)
print(json.dumps(SUMMARY, indent=2, default=str))
if BACKUP_SMALL_ARTIFACTS and MODE in {'full', 'resume', 'verify'}:
    for row in SUMMARY:
        config, run_id = row['config'], row['run_id']
        stage4_dir = paths['stage4_runs_root'] / config / run_id
        if row['stage4_complete']:
            backup_small_artifacts(stage4_dir, paths['backup_root'] / 'stage4' / config)
        for branch in BRANCHES:
            stage5_dir = paths['stage5_runs_root'] / config / run_id / branch
            if row[f'stage5_{branch}_complete']:
                backup_small_artifacts(stage5_dir, paths['backup_root'] / 'stage5' / config / branch)
print('Scientific status: PROVISIONAL_RESEARCH; official reproduction: false')

## LEGACY OLD ORCHESTRATOR CELLS — DO NOT RUN

This cell is metadata-only. It checks the Train footer, schema, expected window count, resource availability, and the single-root rule. It does not read Train feature values.

In [ ]:
preflight = {}
if layout_status['status'] == 'PASS':
    for config in CONFIGS:
        preflight[config] = stage4_metadata_preflight(paths, config)
preflight

## Stage 4 MI — Train-only Top 100

In `dry_run`, `full`, and `resume` mode this section is intentionally skipped or guarded by the configuration above. For a full MI run, use new Run IDs if the listed Windows/Drive run already exists; never overwrite a completed run.

In [ ]:
mi_results = {}
if MODE in {'full', 'resume'}:
    if not CONFIRM_PROVISIONAL_RESEARCH:
        raise RuntimeError('MI full/resume requires provisional confirmation')
    for config in CONFIGS:
        mi_results[config] = run_stage4_mi(
            paths, config, RUN_IDS[config], mode=MODE,
            confirm_provisional_research=CONFIRM_PROVISIONAL_RESEARCH,
        )
elif MODE == 'verify':
    for config in CONFIGS:
        mi_results[config] = verify_stage4_run(paths, config, RUN_IDS[config])
else:
    mi_results = {'status': 'DRY_RUN_SKIPPED_MI_FIT'}
mi_results

## Stage 5A RF-RFE — Train-only full ranking 100 → 1

RF and SVM-RFE are separate branches. Each branch binds to its exact Stage 4 MI run, writes step-one checkpoints, and never opens Validation/Test. XGBoost is intentionally absent and remains BLOCKED.

In [ ]:
rfe_results = {}
if MODE in {'full', 'resume'}:
    if not CONFIRM_PROVISIONAL_RESEARCH:
        raise RuntimeError('RFE full/resume requires provisional confirmation')
    for config in CONFIGS:
        for branch in BRANCHES:
            rfe_results[f'{config}/{branch}'] = run_stage5_rfe(
                paths, config, RUN_IDS[config], branch, mode=MODE,
                git_commit=GIT_COMMIT,
                confirm_provisional_research=CONFIRM_PROVISIONAL_RESEARCH,
            )
elif MODE == 'verify':
    for config in CONFIGS:
        for branch in BRANCHES:
            rfe_results[f'{config}/{branch}'] = run_stage5_rfe(
                paths, config, RUN_IDS[config], branch, mode='verify',
            )
else:
    rfe_results = {'status': 'DRY_RUN_SKIPPED_RFE_FIT'}
rfe_results

## Drive backup and checkpoint integrity

Only small JSON/CSV/Markdown/marker artifacts are backed up. No Parquet dataset file is copied. Backup writes a completion marker last and refuses an existing backup target.

In [ ]:
backup_results = {}
if BACKUP_SMALL_ARTIFACTS and MODE in {'full', 'resume', 'verify'}:
    for config in CONFIGS:
        run_id = RUN_IDS[config]
        stage4_dir = paths['stage4_runs_root'] / config / run_id
        if (stage4_dir / 'MI_COMPLETE.json').exists():
            backup_results[f'{config}/stage4'] = backup_small_artifacts(
                stage4_dir, paths['backup_root'] / 'stage4' / config
            )
        for branch in BRANCHES:
            stage5_dir = paths['stage5_runs_root'] / config / run_id / branch
            if (stage5_dir / 'COMPLETE.json').exists():
                backup_results[f'{config}/stage5/{branch}'] = backup_small_artifacts(
                    stage5_dir, paths['backup_root'] / 'stage5' / config / branch
                )
backup_results

## Four-configuration summary

This summary is marker-based and does not promote a run to official reproduction.

In [ ]:
summary = summarize_configs(paths, RUN_IDS)
summary

## Stage 5B boundary

This notebook does not run Validation k-search, final Train+Validation refit, or Test evaluation. Follow `code_CICFLOW/stage5_rfe/reports/STAGE5B_CONTRACT.md` only after Stage 5A ranking and source compatibility are scientifically locked.